In [1]:
import os

import torch
import pandas as pd
import scanpy as sc
from sklearn import metrics
import multiprocessing as mp
from GraphST import GraphST

# Run device，by default, the package is implemented on 'cpu'. We recommend using GPU.
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')


### Specify Paths

In [2]:
celltype_col =  "celltype_minor" 

sc_path = "/data/Ajita/Spatial/Datasets/Spatial_Deconvolution/Breast_Cancer/SC/CID4290_SC_filtered.h5ad"
results_path = "/data/Ajita/Spatial/Datasets/Spatial_Deconvolution/Breast_Cancer/final_outputs/ER+/" #final_outputs, deconvolution
sc_adata = sc.read_h5ad(sc_path)

st_path = "/data/Ajita/Spatial/Datasets/Spatial_Deconvolution/Breast_Cancer/simulator_outputs/ER+/"

st_adata = sc.read_h5ad(st_path + "simulated_gene_expression_average.h5ad")

st_adata.obs_names_make_unique()
st_adata.var_names = pd.Categorical(st_adata.var_names).astype(str)
st_adata.var_names_make_unique()

sc_adata.obs_names_make_unique()
sc_adata.var_names = pd.Categorical(sc_adata.var_names).astype(str)
sc_adata.var_names_make_unique()

### GraphST preprocessing

In [5]:
# preprocessing for ST data
GraphST.preprocess(st_adata)

# build graph
GraphST.construct_interaction(st_adata)
GraphST.add_contrastive_label(st_adata)

# preprocessing for scRNA data
GraphST.preprocess(sc_adata)

# find overlap genes
from GraphST.preprocess import filter_with_overlap_gene
st_adata, sc_adata = filter_with_overlap_gene(st_adata, sc_adata)

# get features
GraphST.get_feature(st_adata)



Number of overlap genes: 891


/home/ajita/anaconda3/envs/GraphST/lib/python3.9/site-packages/anndata/compat/_overloaded_dict.py:106: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  self.data[key] = value


### GraphST Model

In [6]:
model = GraphST.GraphST(st_adata, sc_adata, epochs = 1200, random_seed = 50, device = device, deconvolution = True)
st_adata, sc_adata = model.train_map()



Begin to train ST data...


100%|████████████████████████████████████████████████████████████████████████████████████| 1200/1200 [00:16<00:00, 70.81it/s]


Optimization finished for ST data!
Begin to train scRNA data...


100%|████████████████████████████████████████████████████████████████████████████████████| 1200/1200 [00:33<00:00, 35.36it/s]


Optimization finished for cell representation learning!
Begin to learn mapping matrix...


100%|████████████████████████████████████████████████████████████████████████████████████| 1200/1200 [01:40<00:00, 11.97it/s]


Mapping matrix learning finished!


In [7]:
# Project cells into spatial space
from GraphST.utils import project_cell_to_spot


sc_adata.obs["cell_type"] = sc_adata.obs[celltype_col]
project_cell_to_spot(st_adata, sc_adata, retain_percent=0.15)
#After projection, the probability distributions of each cell type in spots are saved in adata.obs.

In [8]:
celltypes = sc_adata.obs["cell_type"].unique().tolist()
celltypes.sort()

In [9]:
st_adata.obs[celltypes].to_csv(results_path + "output_GraphST.csv")